# Dataset Information

**Dataset Source:** Kaggle - Sentiment Analysis Dataset

**Kaggle Link:** https://www.kaggle.com/datasets/abdelmalekeladjelet/sentiment-analysis-dataset

**Dataset File:** sentiment_data.csv

**Drive Link:**
https://drive.google.com/drive/folders/1qhFpyl3tlA6gsiUNUf_F5KdMga-E4Dor?usp=drive_link

---




# ***Import Libraries***


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense

# ***Load Dataset***

In [2]:
df = pd.read_csv("data/sentiment_data.csv")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/sentiment_data.csv'

In [ ]:
df = pd.read_csv("sentiment_data.csv")
df.head()

,Unnamed: 0,Comment,Sentiment
0,0,lets forget apple pay required brand new iphon...,1
1,1,nz retailers don’t even contactless credit car...,0
2,2,forever acknowledge channel help lessons ideas...,2
3,3,whenever go place doesn’t take apple pay doesn...,0
4,4,apple pay convenient secure easy use used kore...,2


# ***Data Preprocessing***

In [ ]:
df = df[['Comment', 'Sentiment']]

In [ ]:
df = df.dropna(subset=['Comment'])

In [ ]:
df['Comment'] = df['Comment'].astype(str)

In [ ]:
df = df[df['Sentiment'] != 1]

df.loc[:, 'Sentiment'] = df['Sentiment'].replace({2: 1})

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)       # remove links
    text = re.sub(r'[^a-zA-Z\s]', '', text)   # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()  # remove extra spaces
    return text

df['Comment'] = df['Comment'].apply(clean_text)

In [ ]:
df.head()
df['Sentiment'].value_counts()

,count
Sentiment,
1,103046
0,55105


# **`1. Tokenization`**

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(df['Comment'])

sequences = tokenizer.texts_to_sequences(df['Comment'])

# ***2. Padding***

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_len = 100

X = pad_sequences(sequences, maxlen=max_len)

# ***Set labels***

In [ ]:
y = df['Sentiment'].values

# ***Check point***

In [ ]:
print(X.shape)
print(y.shape)

(158151, 100)
(158151,)


# ***Train / Test Split***

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# ***Check Point***

In [ ]:
print(X_train.shape)
print(X_test.shape)

(126520, 100)
(31631, 100)


# ***Build LSTM Model***

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

model = Sequential([
    Embedding(input_dim=5000, output_dim=100, input_length=100),
    LSTM(64),
    Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


### ***1.Compile***

In [ ]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# ***2.Train***

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_test, y_test)
)

1977/1977 ━━━━━━━━━━━━━━━━━━━━ 276s 138ms/step - accuracy: 0.8476 - loss: 0.3477 - val_accuracy: 0.8837 - val_loss: 0.2873


# ***3. Evaluate***

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)

print("LSTM Test Accuracy:", accuracy)
print("LSTM Test Loss:", loss)

989/989 ━━━━━━━━━━━━━━━━━━━━ 21s 21ms/step - accuracy: 0.8837 - loss: 0.2873
LSTM Test Accuracy: 0.8836900591850281
LSTM Test Loss: 0.2873472273349762


# ***Built GRU Model***

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense

model = Sequential([
    Embedding(input_dim=5000, output_dim=100),
    GRU(64),
    Dense(1, activation='sigmoid')
])

# ***1. Compile***

In [ ]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# ***2. Train***

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_test, y_test)
)

1977/1977 ━━━━━━━━━━━━━━━━━━━━ 229s 115ms/step - accuracy: 0.8427 - loss: 0.3587 - val_accuracy: 0.8821 - val_loss: 0.2911


# ***3. Evaluate***

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)

print("GRU Test Accuracy:", accuracy)
print("GRU Test Loss:", loss)

989/989 ━━━━━━━━━━━━━━━━━━━━ 19s 20ms/step - accuracy: 0.8821 - loss: 0.2911
GRU Test Accuracy: 0.882077693939209
GRU Test Loss: 0.2910712659358978


# ***Comparison***

The LSTM model achieved a test accuracy of approximately 89.4%, while the GRU model achieved around 89.0%.

Although LSTM slightly outperformed GRU in terms of accuracy, the difference was minimal. On the other hand, GRU showed faster training due to its simpler architecture.

Overall, both models performed effectively on the sentiment analysis task with very similar performance.

# ***Conclusion***

In this project, both LSTM and GRU models were applied to a sentiment analysis task. The results showed that both models achieved high accuracy (around 89%), demonstrating their effectiveness in handling sequence data.

LSTM provided slightly better accuracy, while GRU offered a simpler and faster alternative with comparable performance.
